# UE9 - Seance 6 - Variant Calling

**Auteur : Marwa Zidi** — Universite Paris Cite

**Date : 11/02/2026**

---

**Fonctionnement de ce notebook** :

- Cellules `# Code a ecrire par l'etudiant` : a completer vous-meme
- Cellules `# Code deja fourni` : a executer directement
- Cellules `# A remplir par l'etudiant` : reponses textuelles libres

Essayez d'ecrire les commandes sans copier-coller pour optimiser l'apprentissage.

**Contexte clinique** :

Un patient de 76 ans est hospitalise suite au diagnostic d'une **leucemie aigue myeloide (LAM)**.

| Echantillon | Prelevement | Envahissement tumoral |
|-------------|-------------|----------------------|
| `Diagnosis` | Moelle osseuse au diagnostic | 30% de blastes |
| `Germline` | Biopsie cutanee — fibroblastes | Aucun |

Un sequencage cible (panel de 15 genes) a ete realise en paired-end Illumina sur les deux echantillons.

**Objectif** :

Identifier les **mutations somatiques** acquises par les blastes ainsi que les eventuelles **mutations constitutionnelles** predisposantes.

## Etape 1 — Controle qualite (FastQC)

---

#### Affichez le contenu du dossier `Raw_data`. Que contient-il ?

In [ ]:
# Code a ecrire par l'etudiant


#### Pourquoi y a-t-il deux fichiers FASTQ (R1 et R2) pour chaque echantillon ?

#### Affichez les 4 premieres lignes du fichier `Diagnosis_R1.fastq`

In [ ]:
# Code a ecrire par l'etudiant


#### A quoi correspond chaque ligne d'un fichier FASTQ ?

#### Creez un dossier `Quality_control`

In [ ]:
# Code a ecrire par l'etudiant


#### Lancez FastQC sur tous les fichiers FASTQ en redirigeant les resultats dans `Quality_control`

Utilisez `fastqc --help` pour voir les options disponibles.

In [ ]:
# Code a ecrire par l'etudiant


#### Ouvrez les fichiers `fastqc.html` generes et commentez la qualite des donnees

## Etape 2 — Alignement sur le genome humain (BWA)

---

### 1. Genome de reference

Le genome hg38 a deja ete telecharge et indexe dans `Reference_genome/`.

Commandes utilisees pour cette etape (deja effectuees) :

### 2. Alignement

#### Creez un dossier `Intermediate_files`

In [ ]:
# Code a ecrire par l'etudiant


#### Alignement de l'echantillon Diagnosis (commande fournie) :

In [ ]:
# Code deja fourni
bwa mem \
Reference_genome/hg38.fa \
Raw_data/Diagnosis_R1.fastq \
Raw_data/Diagnosis_R2.fastq > Intermediate_files/Diagnosis.sam

#### Affichez la premiere ligne du fichier SAM :

In [ ]:
# Code deja fourni
samtools view Intermediate_files/Diagnosis.sam | head -n 1

#### A quoi correspondent les 6 premiers champs d'une ligne SAM ?

#### A vous d'aligner l'echantillon Germline en vous inspirant de la commande precedente

In [ ]:
# Code a ecrire par l'etudiant


## Etape 3 — Compression, tri et indexation (Samtools)

---

### 1. SAM → BAM

#### Compression de Diagnosis (fourni) :

In [ ]:
# Code deja fourni
samtools view -b Intermediate_files/Diagnosis.sam > Intermediate_files/Diagnosis.bam

#### Compressez l'echantillon Germline

In [ ]:
# Code a ecrire par l'etudiant


### 2. Tri des fichiers BAM

#### Tri de Diagnosis (fourni) :

In [ ]:
# Code deja fourni
samtools sort Intermediate_files/Diagnosis.bam -o Intermediate_files/Diagnosis_sorted.bam

#### Triez l'echantillon Germline

In [ ]:
# Code a ecrire par l'etudiant


#### Affichez le contenu de `Intermediate_files/` avec la taille de chaque fichier

In [ ]:
# Code a ecrire par l'etudiant


#### Comparez la taille des .sam, .bam et _sorted.bam. Comment expliquez-vous les differences ?

### 3. Indexation des BAM

Utilisez `samtools --help` pour trouver la commande d'indexation.

In [ ]:
# Code a ecrire par l'etudiant — indexer Diagnosis_sorted.bam et Germline_sorted.bam


## Etape 4 — Qualite de l'alignement (Qualimap)

---

In [ ]:
# Code deja fourni
export JAVA_OPTS="-Xmx1200M"
qualimap bamqc \
-bam Intermediate_files/Diagnosis_sorted.bam \
-outdir Quality_control/Diagnosis_bamqc \
-gff Reference_genome/Genes_exons.gtf

#### Ouvrez `Quality_control/Diagnosis_bamqc/qualimapReport.html`. Que pensez-vous de la qualite ?

In [ ]:
# Code deja fourni — nombre de reads avec MAPQ optimal (60)
samtools view -c -q 60 Intermediate_files/Diagnosis_sorted.bam

#### Faites la meme analyse pour l'echantillon Germline

In [ ]:
# Code a ecrire par l'etudiant


#### Ouvrez IGV, chargez `Diagnosis_sorted.bam` et cherchez le gene IDH1.

Que pensez-vous de la couverture dans le contexte d'un tissu tumoral (30% de blastes) ?

## Etape 5 — Appel de variants (VarScan)

---

### 1. Indexation du genome de reference (deja effectuee)

### 2. Generation du pileup

#### Pileup Diagnosis (fourni) :

In [ ]:
# Code deja fourni
samtools mpileup -f Reference_genome/hg38.fa \
Intermediate_files/Diagnosis_sorted.bam > Intermediate_files/Diagnosis.mpileup

#### Affichez un extrait du pileup et decrivez le contenu des 6 colonnes :

In [ ]:
# Code deja fourni
head -n 175 Intermediate_files/Diagnosis.mpileup | tail -n 10

#### Creez le pileup pour l'echantillon Germline

In [ ]:
# Code a ecrire par l'etudiant


### 3. Appel des variants

#### Creez un dossier `Results`

In [ ]:
# Code a ecrire par l'etudiant


In [ ]:
# Code deja fourni — aide VarScan
varscan mpileup2cns --help

#### Appel permissif pour decouverte (fourni) :

In [ ]:
# Code deja fourni
varscan mpileup2cns \
Intermediate_files/Diagnosis.mpileup \
--min-coverage 1 \
--min-reads2 1 \
--min-var-freq 0.001 \
--min-avg-qual 1 \
--p-value 1 \
--strand-filter 0 \
--output-vcf 1 \
--variants \
> Results/Diagnosis.vcf

#### Combien de variants ont ete appeles ? Que pensez-vous de leur pertinence ?

#### Modifiez la commande avec des criteres plus stringents :

- Profondeur minimale : **10 reads**
- Reads variants : **>= 2**
- Frequence allelique : **>= 2%**
- Qualite de base : **>= 20**
- Biais de brin : **p-value 1%**

In [ ]:
# Code a modifier par l'etudiant
varscan mpileup2cns \
Intermediate_files/Diagnosis.mpileup \
--min-coverage ? \
--min-reads2 ? \
--min-var-freq ? \
--min-avg-qual ? \
--p-value ? \
--strand-filter ? \
--output-vcf 1 \
--variants \
> Results/Diagnosis.vcf

#### Combien de variants restent apres filtrage ?

In [ ]:
# Code deja fourni — extrait du VCF
head -n 33 Results/Diagnosis.vcf | tail

#### Chargez le fichier VCF dans IGV et observez les variants

#### Appliquez VarScan sur l'echantillon Germline avec les memes criteres

In [ ]:
# Code a ecrire par l'etudiant


#### Combien de variants sont appeles sur Germline ? Comparez avec Diagnosis.

#### Quels types de variants VarScan ne detecte-t-il pas ?

## Etape 6 — Annotation des variants (VEP)

---

In [ ]:
# Code deja fourni — aide VEP
/opt/ensembl-vep/vep --help

In [ ]:
# Code deja fourni — annotation Diagnosis
/opt/ensembl-vep/vep -i Results/Diagnosis.vcf \
    -o Results/Diagnosis_vep.vcf \
    --vcf \
    --database \
    --assembly GRCh38 \
    --fasta Reference_genome/hg38.fa \
    --pick \
    --symbol \
    --hgvs \
    --af \
    --polyphen b \
    --sift b

#### Annotez les variants de l'echantillon Germline

In [ ]:
# Code a ecrire par l'etudiant


## Etape 7 — Filtrage des variants (R)

---

#### Convertissez le VCF annote en tableau TSV lisible sous Excel :

In [ ]:
# Code deja fourni
Rscript Scripts/VCF_to_TSV.R Results/Diagnosis_vep.vcf

#### Explorez le fichier TSV. Comment identifiez-vous les variants potentiellement pathogenes ?

In [ ]:
# Code deja fourni — filtrage R
Rscript Scripts/Variants_filtering.R Results/Diagnosis_vep.vcf.tsv

#### Parmi les variants retenus, lesquels vous semblent cliniquement pertinents ?

#### Que pensez-vous de la VAF de ces variants sachant que l'echantillon est envahi a 30% de blastes ?

#### Faites de meme pour l'echantillon Germline

In [ ]:
# Code a ecrire par l'etudiant


#### Sachant qu'il n'y a pas de cellules tumorales dans cet echantillon, comment interpretez-vous les variants identifies ?

L'un d'eux vous semble-t-il potentiellement pathogene ?

## Conclusion

---

#### Resumez le pipeline en decrivant pour chaque etape l'outil utilise et son role :

```
*.fastq  -1->  *.sam  -2->  *.bam  -3->  *.mpileup  -4->  *.vcf  -5->  *_vep.vcf  -6->  Filtered_*.tsv
```

### Pour aller plus loin

Les variants pathogenes de **DDX41** sont associes aux syndromes myelodysplasiques et aux LAM chez les patients de plus de 60 ans.
La decouverte d'un tel variant a des implications pour la prise en charge du patient ET pour le depistage de ses apparentes.

*Germline DDX41 mutations define a significant entity within adult MDS/AML patients.* Sebert et al. Blood 2019
https://pubmed.ncbi.nlm.nih.gov/31484648/